# Databricks Workspace, Catalog & Compute

**Course 1, Week 1: Lakehouse Architecture & Platform**

This notebook demonstrates the core Databricks workspace components:
Workspace navigation, Unity Catalog hierarchy, and compute resources.

**Certification Alignment:** Databricks Accredited Lakehouse Platform Fundamentals

## 1. Workspace Overview

The Databricks Workspace is your central hub:
- **Home:** Personal notebooks and files
- **Workspace:** Shared notebooks, libraries, repos
- **Catalog:** Browse data assets (Unity Catalog)
- **Workflows:** Scheduled jobs and pipelines
- **Compute:** Clusters and SQL Warehouses
- **Experiments:** MLflow tracking (Course 3)


## 2. Unity Catalog Hierarchy

Unity Catalog organizes data in a three-level namespace:

```
Metastore (top-level container)
└── Catalog (logical grouping — e.g., "production", "dev")
        ├── View (virtual table from query)
        └── Function (UDFs)
```

**Key terms for certification:**
- **Metastore:** Top-level container for all data assets
- **Catalog:** Logical grouping (like a database server)
- **Schema:** Collection of tables/views (like a database)

## 3. Explore Available Catalogs

In [0]:
# List catalogs (Unity Catalog)
try:
    catalogs = spark.sql("SHOW CATALOGS")
    catalogs.show()
except Exception as e:
    print(f"Unity Catalog may not be enabled: {e}")
    print("On Databricks Free Edition, use the default catalog: 'hive_metastore'")

+---------+
|  catalog|
+---------+
|  samples|
|   system|
|workspace|
+---------+



## 4. Browse Schemas and Tables

In [0]:
# List schemas in the default catalog
try:
    schemas = spark.sql("SHOW SCHEMAS")
    schemas.show(truncate=False)
except Exception as e:
    print(f"Error listing schemas: {e}")

+------------------+
|databaseName      |
+------------------+
|default           |
|information_schema|
+------------------+



## 5. Compute Resources

Databricks offers several compute options:

| Compute Type | Best For | Key Feature |
|-------------|----------|-------------|
| **All-Purpose Cluster** | Interactive development | Shared, configurable |
| **Job Cluster** | Scheduled workloads | Auto-created, auto-terminated |
| **SQL Warehouse** | BI and SQL analytics | Optimized for SQL, Photon |
| **Serverless** | On-demand compute | No cluster management |

On **Databricks Free Edition**, you get a single-node cluster for development.

In [0]:
# Check current cluster configuration
print(f"Spark Version: {spark.version}")

try:
    print(f"Cluster ID: {spark.conf.get('spark.databricks.clusterUsageTags.clusterId')}")
except Exception:
    print("Cluster ID: N/A (Restricted)")

try:
    print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
except Exception:
    print("Driver Memory: N/A (Restricted)")

try:
    print(f"Executor Cores: {spark.conf.get('spark.executor.cores')}")
except Exception:
    print("Executor Cores: N/A (Restricted)")

Spark Version: 4.2.0
Cluster ID: 0901-094617-pjq8vn9q-v2n
Driver Memory: N/A (Restricted)
Executor Cores: N/A (Restricted)


## 6. Data Governance with Unity Catalog

Unity Catalog provides a **single governance solution** across all workloads:

- **Access Control:** Fine-grained permissions on catalogs, schemas, tables
- **Data Lineage:** Track data flow from source to consumption
- **Auditing:** Full audit trail of who accessed what data
- **Data Sharing:** Delta Sharing for cross-organization sharing

**Certification note:** Unity Catalog is the recommended governance layer
for all Databricks deployments.

## 7. DBFS (Databricks File System)

DBFS provides a distributed file system mounted to the workspace:

In [0]:
# Explore DBFS
try:
    files = dbutils.fs.ls("/")
    for f in files[:10]:
        print(f"  {f.name:30s} {'DIR' if f.isDir() else f'{f.size:>10,} bytes'}")
except NameError:
    print("dbutils not available — run this notebook on Databricks")

  Volumes/                       DIR
  Workspace/                     DIR
  databricks-datasets/           DIR


## 8. Summary

| Component | Purpose | Certification Focus |
|-----------|---------|-------------------|
| Workspace | Central development hub | Navigation, collaboration |
| Unity Catalog | Data governance | Metastore/Catalog/Schema hierarchy |
| Compute | Processing resources | Cluster types, SQL Warehouses |
| DBFS | File storage | Data access patterns |


## Create Your Own Schema

EXERCISE: Create a schema and table in the catalog.

In [0]:
%sql

-- EXERCISE: Create a schema for this lab
-- YOUR CODE HERE (Hint: CREATE SCHEMA IF NOT EXISTS lab_workspace)

CREATE SCHEMA IF NOT EXISTS workspace.lab_workspace;

In [0]:
%sql
-- EXERCISE: Create a table in your schema
-- Create a table called lab_workspace.cities with columns: name, state, population
-- YOUR CODE HERE

CREATE TABLE IF NOT EXISTS workspace.lab_workspace.cities (
    name STRING,
    state STRING,
    population INT
);

In [0]:
%sql
-- EXERCISE: Insert at least 3 cities
-- YOUR CODE HERE

INSERT INTO workspace.lab_workspace.cities VALUES
    ('Seattle', 'Washington', 733778),
    ('Austin', 'Texas', 961855),
    ('San Francisco', 'California', 808437);

num_affected_rows,num_inserted_rows
3,3


In [0]:
 %sql
-- EXERCISE: Query your table
-- YOUR CODE HERE

SELECT * FROM workspace.lab_workspace.cities;



name,state,population
Seattle,Washington,733778
Austin,Texas,961855
San Francisco,California,808437


## File System Exploration

EXERCISE: Explore DBFS and Databricks sample datasets.

In [0]:
# EXERCISE: List the contents of /databricks-datasets/
# Hint: Use dbutils.fs.ls() or %fs magic command
# YOUR CODE HERE

# List the contents of the root sample datasets folder
display(dbutils.fs.ls("/databricks-datasets/"))

path,name,size,modificationTime
dbfs:/databricks-datasets/COVID/,COVID/,0,1788262022988
dbfs:/databricks-datasets/README.md,README.md,976,1596557781000
dbfs:/databricks-datasets/Rdatasets/,Rdatasets/,0,1788262022988
dbfs:/databricks-datasets/SPARK_README.md,SPARK_README.md,3359,1596557823000
dbfs:/databricks-datasets/adult/,adult/,0,1788262022988
dbfs:/databricks-datasets/airlines/,airlines/,0,1788262022988
dbfs:/databricks-datasets/amazon/,amazon/,0,1788262022988
dbfs:/databricks-datasets/asa/,asa/,0,1788262022988
dbfs:/databricks-datasets/atlas_higgs/,atlas_higgs/,0,1788262022988
dbfs:/databricks-datasets/bikeSharing/,bikeSharing/,0,1788262022988


In [0]:
# see all metadata rows for your table, including its location
display(spark.sql("DESCRIBE TABLE EXTENDED workspace.lab_workspace.cities"))

col_name,data_type,comment
name,string,null
state,string,null
population,int,null
,,
# Delta Statistics Columns,,
Column Names,"name, state, population",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,workspace,


In [0]:
# EXERCISE: Find and preview one sample dataset
# Hint: Use dbutils.fs.head() to preview the first few bytes
# YOUR CODE HERE

file_path = "/databricks-datasets/Rdatasets/data-001/csv/ggplot2/diamonds.csv"

# Preview only the first 500 bytes
print(dbutils.fs.head(file_path, max_bytes=500))

[Truncated to first 500 bytes]
"","carat","cut","color","clarity","depth","table","price","x","y","z"
"1",0.23,"Ideal","E","SI2",61.5,55,326,3.95,3.98,2.43
"2",0.21,"Premium","E","SI1",59.8,61,326,3.89,3.84,2.31
"3",0.23,"Good","E","VS1",56.9,65,327,4.05,4.07,2.31
"4",0.29,"Premium","I","VS2",62.4,58,334,4.2,4.23,2.63
"5",0.31,"Good","J","SI2",63.3,58,335,4.34,4.35,2.75
"6",0.24,"Very Good","J","VVS2",62.8,57,336,3.94,3.96,2.48
"7",0.24,"Very Good","I","VVS1",62.3,57,336,3.95,3.98,2.47
"8",0.26,"Very Good","H","SI1",61.9,55,3


## Compute Information

EXERCISE: Inspect your current cluster.

In [0]:
# Check current cluster configuration
print(f"Spark Version: {spark.version}")

try:
    print(f"Cluster ID: {spark.conf.get('spark.databricks.clusterUsageTags.clusterId')}")
except Exception:
    print("Cluster ID: N/A (Restricted)")

try:
    print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
except Exception:
    print("Driver Memory: N/A (Restricted)")

try:
    print(f"Executor Cores: {spark.conf.get('spark.executor.cores')}")
except Exception:
    print("Executor Cores: N/A (Restricted)")

Spark Version: 4.2.0
Cluster ID: 0901-094617-pjq8vn9q-v2n
Driver Memory: N/A (Restricted)
Executor Cores: N/A (Restricted)


## Validation

In [0]:
def validate_lab():
    """Validate lab completion."""
    checks = []

    # Check 1: Schema exists
    try:
        spark.sql("SHOW TABLES IN lab_workspace")
        checks.append(("Schema created", True))
    except Exception:
        checks.append(("Schema created", False))

    # Check 2: Cities table exists with data
    try:
        df = spark.sql("SELECT * FROM lab_workspace.cities")
        checks.append(("Cities table with data", df.count() >= 3))
    except Exception:
        checks.append(("Cities table with data", False))

    print("Lab Validation Results:")
    print("-" * 40)
    all_passed = True
    for name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed:
            all_passed = False

    if all_passed:
        print("\nAll checks passed! Lab complete.")
    else:
        print("\nSome checks failed. Review your code above.")

validate_lab()


Lab Validation Results:
----------------------------------------
  [PASS] Schema created
  [PASS] Cities table with data

All checks passed! Lab complete.


In [0]:
# Clean up
try:
    print("Attempting to drop schema 'workspace.lab_workspace'...")
    spark.sql("DROP SCHEMA IF EXISTS workspace.lab_workspace CASCADE")
    print("SUCCESS: Schema 'workspace.lab_workspace' and all its contents were successfully deleted.")
except Exception as e:
    print(f"NOTICE: Could not drop schema. Reason: {e}")

Attempting to drop schema 'workspace.lab_workspace'...
SUCCESS: Schema 'workspace.lab_workspace' and all its contents were successfully deleted.
